# FDA Drug Adverse Event Severity Predictor
## Notebook 2: Preprocessing & Feature Engineering

**Goal:** Clean the data, encode features, and split into train/test sets ready for modeling.

**Key decisions made from EDA:**
- Target: `serious_binary` (1 = Yes, 0 = No)
- Drop outcome leakage columns (`is_fatal`, `is_hospitalized`, `is_life_threat`, `is_disabling`) — these happen AFTER the event is already serious, so using them would be cheating
- Drop identifier/date columns that carry no predictive signal
- Fill missing numerics with median, missing categoricals with 'Unknown'
- Encode categoricals with Label Encoding
- Scale numerics with StandardScaler

---
## 1. Import Libraries & Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

df = pd.read_csv('fda_adverse_events_2015_2026_CLEAN.csv', low_memory=False)
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

Loaded: 528,000 rows x 30 columns


---
## 2. Create Target Variable

In [3]:
# Map serious Yes/No to 1/0
df['serious_binary'] = df['serious'].map({'Yes': 1, 'No': 0})

print('Target variable created:')
print(df['serious_binary'].value_counts())
print(f'\nClass balance: {df["serious_binary"].mean()*100:.1f}% serious')

Target variable created:
serious_binary
1    395000
0    133000
Name: count, dtype: int64

Class balance: 74.8% serious


---
## 3. Drop Columns We Cannot Use

**Leakage columns**: these are outcomes that only exist BECAUSE the event was serious. Using them would let the model "cheat" and it would fail in the real world:
- `is_fatal`, `is_hospitalized`, `is_life_threat`, `is_disabling`, `serious_flags`, `reaction_outcomes`

**Identifier / date columns**: no predictive signal:
- `report_id`, `receive_date`, `report_age_days`

**Target source column**: already encoded into `serious_binary`:
- `serious`

**High cardinality text columns**: too many unique values to encode meaningfully:
- `reactions`, `brand_name`, `manufacturer`, `drug_indication`

In [5]:
drop_cols = [
    # Leakage — outcome columns
    'is_fatal', 'is_hospitalized', 'is_life_threat', 'is_disabling',
    'serious_flags', 'reaction_outcomes', 'patient_recovered',
    # Identifiers / dates
    'report_id', 'receive_date', 'report_age_days',
    # Target source
    'serious',
    # High cardinality free text
    'reactions', 'brand_name', 'manufacturer', 'drug_indication'
]

df_model = df.drop(columns=drop_cols)

print(f'Columns remaining: {df_model.shape[1]}')
print(df_model.columns.tolist())

Columns remaining: 16
['year', 'month', 'quarter', 'primary_reaction', 'num_reactions', 'suspect_drug', 'drug_route', 'pharm_class', 'num_drugs', 'drug_count_category', 'patient_age_years', 'age_group', 'patient_sex', 'patient_weight_kg', 'country', 'serious_binary']


---
## 4. Handle Missing Values

In [7]:
# Check remaining missing values
missing = df_model.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0])

Missing values per column:
patient_age_years    151509
patient_weight_kg    379923
dtype: int64


In [9]:
# Fill numeric columns with median
numeric_cols = df_model.select_dtypes(include=['float64', 'int64']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'serious_binary']

for col in numeric_cols:
    if df_model[col].isnull().sum() > 0:
        median_val = df_model[col].median()
        df_model[col] = df_model[col].fillna(median_val)
        print(f'  Filled {col} with median: {median_val:.2f}')

# Fill categorical/object columns with 'Unknown'
object_cols = df_model.select_dtypes(include=['object']).columns.tolist()

for col in object_cols:
    if df_model[col].isnull().sum() > 0:
        df_model[col] = df_model[col].fillna('Unknown')
        print(f'  Filled {col} with Unknown')

print(f'\nMissing values remaining: {df_model.isnull().sum().sum()}')

  Filled patient_age_years with median: 59.00
  Filled patient_weight_kg with median: 72.56

Missing values remaining: 0


---
## 5. Encode Categorical Features

Machine learning models need numbers, not text. We use **Label Encoding**: it converts each unique category to a number (e.g. Male=0, Female=1, Unknown=2).

In [11]:
le = LabelEncoder()

# Encode all object columns
for col in object_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    print(f'  Encoded: {col}')

# Convert boolean columns to int
bool_cols = df_model.select_dtypes(include=['bool']).columns.tolist()
for col in bool_cols:
    df_model[col] = df_model[col].astype(int)
    print(f'  Converted bool: {col}')

print('\nAll columns are now numeric:')
print(df_model.dtypes)

  Encoded: quarter
  Encoded: primary_reaction
  Encoded: suspect_drug
  Encoded: drug_route
  Encoded: pharm_class
  Encoded: drug_count_category
  Encoded: age_group
  Encoded: patient_sex
  Encoded: country

All columns are now numeric:
year                     int64
month                    int64
quarter                  int32
primary_reaction         int32
num_reactions            int64
suspect_drug             int32
drug_route               int32
pharm_class              int32
num_drugs                int64
drug_count_category      int32
patient_age_years      float64
age_group                int32
patient_sex              int32
patient_weight_kg      float64
country                  int32
serious_binary           int64
dtype: object


---
## 6. Define Features (X) and Target (y)

In [13]:
X = df_model.drop(columns=['serious_binary'])
y = df_model['serious_binary']

print(f'Features (X): {X.shape}')
print(f'Target  (y): {y.shape}')
print(f'\nFeature list:')
print(X.columns.tolist())

Features (X): (528000, 15)
Target  (y): (528000,)

Feature list:
['year', 'month', 'quarter', 'primary_reaction', 'num_reactions', 'suspect_drug', 'drug_route', 'pharm_class', 'num_drugs', 'drug_count_category', 'patient_age_years', 'age_group', 'patient_sex', 'patient_weight_kg', 'country']


---
## 7. Train / Test Split

We split 80% for training, 20% for testing. `random_state=42` makes it reproducible every time you run this, you get the same split. `stratify=y` ensures both splits have the same class balance (74.8% serious).

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set:   {X_train.shape[0]:,} rows')
print(f'Test set:       {X_test.shape[0]:,} rows')
print(f'\nClass balance in training set:')
print(y_train.value_counts(normalize=True).mul(100).round(2))
print(f'\nClass balance in test set:')
print(y_test.value_counts(normalize=True).mul(100).round(2))

Training set:   422,400 rows
Test set:       105,600 rows

Class balance in training set:
serious_binary
1    74.81
0    25.19
Name: proportion, dtype: float64

Class balance in test set:
serious_binary
1    74.81
0    25.19
Name: proportion, dtype: float64


---
## 8. Scale Numeric Features

**Why?** Features like `patient_age_years` (0–100) and `num_drugs` (1–50) are on very different scales. StandardScaler brings them all to the same scale (mean=0, std=1) so the model doesn't treat age as more important just because the numbers are bigger.

**Important:** We fit the scaler ONLY on training data, then apply it to both. This prevents data leakage from the test set.

In [19]:
cols_to_scale = ['patient_age_years', 'patient_weight_kg', 'num_reactions', 'num_drugs', 'report_age_days'] 
# Only scale columns that actually exist in X
cols_to_scale = [c for c in cols_to_scale if c in X_train.columns]

scaler = StandardScaler()
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale]  = scaler.transform(X_test[cols_to_scale])

print(f'Scaled columns: {cols_to_scale}')
print('\nSample of scaled training data:')
X_train[cols_to_scale].describe().round(3)

Scaled columns: ['patient_age_years', 'patient_weight_kg', 'num_reactions', 'num_drugs']

Sample of scaled training data:


,patient_age_years,patient_weight_kg,num_reactions,num_drugs
count,422400.000,422400.000,422400.000,422400.000
mean,-0.000,-0.000,-0.000,-0.000
std,1.000,1.000,1.000,1.000
min,-3.425,-5.235,-0.626,-0.403
25%,-0.291,-0.035,-0.508,-0.350
50%,0.131,-0.035,-0.270,-0.245
75%,0.553,-0.035,0.086,0.070
max,3.809,16.275,29.303,86.104


---
## 9. Save Preprocessed Data

In [21]:
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print('Saved:')
print('  X_train.csv')
print('  X_test.csv')
print('  y_train.csv')
print('  y_test.csv')
print('\nPreprocessing complete. Ready for modeling.')

Saved:
  X_train.csv
  X_test.csv
  y_train.csv
  y_test.csv

Preprocessing complete. Ready for modeling.


---
## 10. Preprocessing Summary

| Step | Decision | Reason |
|------|----------|--------|
| Drop leakage cols | `is_fatal`, `is_hospitalized`, etc. | These are outcomes, not predictors |
| Drop identifiers | `report_id`, `receive_date` | No predictive value |
| Fill numeric nulls | Median imputation | Robust to outliers |
| Fill categorical nulls | 'Unknown' | Preserves missingness as a category |
| Encode categoricals | Label Encoding | Converts text to numbers |
| Train/test split | 80/20, stratified | Preserves class balance |
| Scale numerics | StandardScaler on train only | Prevents data leakage |

**✅ Next Step:** Open `03_model.ipynb` to train Logistic Regression and Random Forest models.